# Step 8B T4x2 Runner — Soft Relation Diagnostics

This notebook **does not edit source code**. It pulls the public GitHub repo, prepares RegDB, and runs two independent diagnostic jobs on Kaggle T4x2:

- GPU 0: baseline CRE + soft relation diagnostics
- GPU 1: UPR-CRE v0.2 best config + soft relation diagnostics

The goal is to check whether the soft relation distribution produced by UPR-CRE v0.2 is better than the original CRE distribution before implementing any soft loss.

## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [1]:
from pathlib import Path
from datetime import datetime

CFG = {
    "GITHUB_OWNER": "TranTruongMMCII",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",

    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",
    "PHASE1_CKPT_PATH": "/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth",

    "SEED": 1,
    "TRIAL": 1,
    "STAGE2_EPOCH": 5,
    "MILESTONES": "8 12",
    "BATCH_PIDNUM": 5,
    "PID_NUMSAMPLE": 4,
    "TEST_BATCH": 64,
    "NUM_WORKERS": 2,
    "LR": 0.00045,

    # Soft relation diagnostic settings
    "SOFT_TOPK": 3,
    "SOFT_TEMP": 0.5,
    "SOFT_START_EPOCH": 2,

    # UPR-CRE v0.2 best config from Step 7
    "UPR_BETA": 0.1,
    "UPR_GAMMA": 0.0,
    "UPR_WARMUP_EPOCH": 2,
    "UPR_FILTER_START_RATIO": 0.55,
    "UPR_FILTER_END_RATIO": 1.0,
    "UPR_FILTER_START_EPOCH": 2,
    "UPR_FILTER_END_EPOCH": 10,
    "UPR_FILTER_MIN_PAIRS": 40,

    # Optional old git-push backup. This pushes small logs/CSV only, not .pth.
    "PUSH_SMALL_BACKUP_TO_GIT": True,
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "GIT_USER_NAME": "Kaggle Bot",
    "GIT_USER_EMAIL": "kaggle-bot@example.com",
}

RUN_SUFFIX = datetime.utcnow().strftime("step8b_t4x2_%Y%m%d_%H%M%S")
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
print("RUN_SUFFIX =", RUN_SUFFIX)
print("repo_dir =", repo_dir)
print("wsl_dir =", wsl_dir)
print("checkpoint exists:", Path(CFG["PHASE1_CKPT_PATH"]).exists())

RUN_SUFFIX = step8b_t4x2_20260625_031837
repo_dir = /kaggle/working/UIT.CS2309
wsl_dir = /kaggle/working/UIT.CS2309/WSL_ReID
checkpoint exists: True


/tmp/ipykernel_58/2966378448.py:46: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_SUFFIX = datetime.utcnow().strftime("step8b_t4x2_%Y%m%d_%H%M%S")


## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [2]:
import subprocess, shutil
from pathlib import Path

repo_url = f"https://github.com/{CFG['GITHUB_OWNER']}/{CFG['REPO_NAME']}.git"
repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]

if repo_dir.exists():
    print("Repo exists. Fetching latest branch...")
    subprocess.run(["git", "fetch", "origin", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", CFG["BRANCH"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{CFG['BRANCH']}"], cwd=repo_dir, check=True)
else:
    subprocess.run(["git", "clone", "--branch", CFG["BRANCH"], repo_url, str(repo_dir)], check=True)

subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, check=True)
subprocess.run(["git", "status", "--short"], cwd=repo_dir, check=True)

Cloning into '/kaggle/working/UIT.CS2309'...


6b875c6


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## Block 2 — Install Kaggle requirements and prepare RegDB

This uses the scripts already committed in your repo. It also checks that two GPUs are visible.

In [3]:
import subprocess
from pathlib import Path

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"

subprocess.run("pip install -q -r requirements-kaggle.txt", cwd=wsl_dir, shell=True, check=True)
subprocess.run(["python", "scripts/apply_kaggle_compat_patches.py"], cwd=wsl_dir, check=True)

prepare_cmd = [
    "python", "scripts/prepare_regdb_kaggle.py",
    "--data-root", CFG["DATA_ROOT"],
]
if CFG["REGDB_SOURCE"]:
    prepare_cmd += ["--regdb-source", CFG["REGDB_SOURCE"]]
subprocess.run(prepare_cmd, cwd=wsl_dir, check=True)
subprocess.run(["python", "scripts/check_kaggle_env.py", "--data-root", CFG["DATA_ROOT"]], cwd=wsl_dir, check=True)
subprocess.run(["nvidia-smi"], check=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset

CompletedProcess(args=['nvidia-smi'], returncode=0)

## Block 3 — Static checks

This checks the code/scripts needed for Step 5. It does not check markdown docs.

In [5]:
from pathlib import Path
import subprocess

wsl_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"] / "WSL_ReID"

required_files = [
    "main.py",
    "wsl.py",
    "relation_metrics.py",
    "soft_relation_metrics.py",
    "scripts/collect_relation_stats.py",
    "scripts/prepare_regdb_kaggle.py",
]
missing = [p for p in required_files if not (wsl_dir / p).exists()]
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))

checks = {
    "main.py": ["--upr-soft-rel", "--upr-soft-topk", "--upr-soft-temp", "--upr-soft-start-epoch"],
    "wsl.py": ["soft_r2i_matrix", "soft_i2r_matrix", "_update_soft_relation_matrix", "_topk_softmax_np"],
    "relation_metrics.py": ["compute_soft_relation_diagnostics", "soft_relation"],
    "soft_relation_metrics.py": ["compute_soft_relation_diagnostics", "topk_accuracy", "correct_mass_mean"],
    "scripts/collect_relation_stats.py": ["soft_r2i_top1_acc", "soft_i2r_topk_acc", "--csv-output"],
}
for rel, markers in checks.items():
    text = (wsl_dir / rel).read_text()
    for marker in markers:
        if marker not in text:
            raise RuntimeError(f"Missing marker {marker!r} in {rel}")

subprocess.run([
    "python", "-m", "py_compile",
    "main.py", "wsl.py", "relation_metrics.py", "soft_relation_metrics.py", "scripts/collect_relation_stats.py"
], cwd=wsl_dir, check=True)
print("Step 8B static checks OK.")

Step 8B static checks OK.


## 4. Run two jobs on T4x2

This uses two GPUs meaningfully:

- GPU 0: original baseline CRE + soft relation diagnostics.
- GPU 1: UPR-CRE v0.2 best filtering curriculum + soft relation diagnostics.

Both runs use the same phase-1 checkpoint and same training schedule. Only UPR/filtering differs.

In [ ]:
import os, subprocess, json
from pathlib import Path

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
phase1_ckpt = Path(CFG["PHASE1_CKPT_PATH"])
assert phase1_ckpt.exists(), f"Missing checkpoint: {phase1_ckpt}"

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode().strip()
run_logs_dir = Path("/kaggle/working/run_logs")
run_logs_dir.mkdir(parents=True, exist_ok=True)

configs = [
    {
        "tag": "baseline_softdiag_p2s5",
        "gpu": 0,
        "enable_upr": False,
        "enable_filter": False,
    },
    {
        "tag": "upr_v02_softdiag_filter055_p2s5",
        "gpu": 1,
        "enable_upr": True,
        "enable_filter": True,
    },
]

jobs = []

def build_cmd(cfg):
    save_path = f"{cfg['tag']}_{RUN_SUFFIX}_{commit}"
    relation_stats_dir = f"../saved_regdb_resnet/{save_path}_{CFG['TRIAL']}/relation_stats"
    cmd = [
        "python", "main.py",
        "--dataset", "regdb",
        "--data-path", CFG["DATA_ROOT"],
        "--debug", "wsl",
        "--save-path", save_path,
        "--arch", "resnet",
        "--trial", str(CFG["TRIAL"]),
        "--seed", str(CFG["SEED"]),
        "--model-path", str(phase1_ckpt),
        "--stage1-epoch", "0",
        "--stage2-epoch", str(CFG["STAGE2_EPOCH"]),
        "--lr", str(CFG["LR"]),
        "--batch-pidnum", str(CFG["BATCH_PIDNUM"]),
        "--pid-numsample", str(CFG["PID_NUMSAMPLE"]),
        "--test-batch", str(CFG["TEST_BATCH"]),
        "--num-workers", str(CFG["NUM_WORKERS"]),
        "--device", "0",
        "--milestones", *CFG["MILESTONES"].split(),
        "--upr-soft-rel",
        "--upr-soft-topk", str(CFG["SOFT_TOPK"]),
        "--upr-soft-temp", str(CFG["SOFT_TEMP"]),
        "--upr-soft-start-epoch", str(CFG["SOFT_START_EPOCH"]),
        "--save-relation-stats",
        "--relation-stats-dir", relation_stats_dir,
    ]
    if cfg["enable_upr"]:
        cmd += [
            "--upr-cre",
            "--upr-beta", str(CFG["UPR_BETA"]),
            "--upr-gamma", str(CFG["UPR_GAMMA"]),
            "--upr-margin-weight", "1.0",
            "--upr-warmup-epoch", str(CFG["UPR_WARMUP_EPOCH"]),
        ]
    if cfg["enable_filter"]:
        cmd += [
            "--upr-filter",
            "--upr-filter-start-epoch", str(CFG["UPR_FILTER_START_EPOCH"]),
            "--upr-filter-end-epoch", str(CFG["UPR_FILTER_END_EPOCH"]),
            "--upr-filter-start-ratio", str(CFG["UPR_FILTER_START_RATIO"]),
            "--upr-filter-end-ratio", str(CFG["UPR_FILTER_END_RATIO"]),
            "--upr-filter-min-pairs", str(CFG["UPR_FILTER_MIN_PAIRS"]),
        ]
    return cmd, save_path, relation_stats_dir

for cfg in configs:
    cmd, save_path, relation_stats_dir = build_cmd(cfg)
    log_path = run_logs_dir / f"{save_path}.log"
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(cfg["gpu"])
    print("\nStarting", cfg["tag"], "on GPU", cfg["gpu"])
    print("save_path:", save_path)
    print("log:", log_path)
    print("cmd:", " ".join(cmd))
    f = open(log_path, "w")
    proc = subprocess.Popen(cmd, cwd=wsl_dir, stdout=f, stderr=subprocess.STDOUT, env=env)
    jobs.append({
        "tag": cfg["tag"],
        "gpu": cfg["gpu"],
        "pid": proc.pid,
        "process": proc,
        "save_path": save_path,
        "relation_stats_dir": relation_stats_dir,
        "log_path": str(log_path),
        "enable_upr": cfg["enable_upr"],
        "enable_filter": cfg["enable_filter"],
    })

runtime_info = {
    "run_suffix": RUN_SUFFIX,
    "commit": commit,
    "phase1_ckpt": str(phase1_ckpt),
    "soft_topk": CFG["SOFT_TOPK"],
    "soft_temp": CFG["SOFT_TEMP"],
    "jobs": [{k: v for k, v in job.items() if k != "process"} for job in jobs],
}
Path("/kaggle/working/step8b_t4x2_runtime_info.json").write_text(json.dumps(runtime_info, indent=2))
print("\nStarted jobs:")
for job in jobs:
    print(job["tag"], "pid=", job["pid"], "gpu=", job["gpu"])

## 5. Monitor

In [ ]:
from pathlib import Path
import subprocess, json

subprocess.run(["nvidia-smi"], check=False)
info = json.loads(Path("/kaggle/working/step8b_t4x2_runtime_info.json").read_text())
for job in info["jobs"]:
    print("\n===", job["tag"], "===")
    log_path = Path(job["log_path"])
    if log_path.exists():
        subprocess.run(["tail", "-40", str(log_path)], check=False)
    else:
        print("Log not found yet:", log_path)

## 6. Wait for jobs

In [ ]:
if "jobs" in globals():
    for job in jobs:
        code = job["process"].wait()
        print(job["tag"], "finished with", code)
else:
    print("jobs variable not found. Check manually:")
    subprocess.run("ps -ef | grep 'python main.py' | grep -v grep", shell=True)

## 7. Collect metrics and soft relation summaries

In [ ]:
from pathlib import Path
import subprocess, json, re, csv

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
info = json.loads(Path("/kaggle/working/step8b_t4x2_runtime_info.json").read_text())

rows = []

def parse_best_metrics(text):
    pattern = re.compile(r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)")
    matches = pattern.findall(text)
    if not matches:
        return "", "", "", "", ""
    best = max(matches, key=lambda x: float(x[3]))
    return best

for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_regdb_resnet" / f"{job['save_path']}_{CFG['TRIAL']}"
    stats_dir = run_dir / "relation_stats"
    stats_csv = stats_dir / "relation_stats_summary.csv"
    log_file = run_dir / "log" / "log.txt"
    print("Collecting", job["tag"], run_dir)
    subprocess.run([
        "python", "scripts/collect_relation_stats.py",
        "--stats-dir", str(stats_dir),
        "--csv-output", str(stats_csv),
    ], cwd=wsl_dir, check=True)
    log_text = log_file.read_text() if log_file.exists() else ""
    r1, r10, r20, map_, minp = parse_best_metrics(log_text)
    last_stats = {}
    if stats_csv.exists():
        with stats_csv.open() as f:
            rr = list(csv.DictReader(f))
            if rr:
                last_stats = rr[-1]
    rows.append({
        "run": job["tag"],
        "save_path": job["save_path"],
        "enable_upr": job["enable_upr"],
        "enable_filter": job["enable_filter"],
        "stage2_epoch": CFG["STAGE2_EPOCH"],
        "soft_topk": info["soft_topk"],
        "soft_temp": info["soft_temp"],
        "best_r1_by_map": r1,
        "best_map": map_,
        "best_minp": minp,
        "last_common_accuracy": last_stats.get("common_accuracy", ""),
        "last_mutual_match_ratio": last_stats.get("mutual_match_ratio", ""),
        "soft_r2i_top1_acc": last_stats.get("soft_r2i_top1_acc", ""),
        "soft_r2i_topk_acc": last_stats.get("soft_r2i_topk_acc", ""),
        "soft_r2i_correct_mass_mean": last_stats.get("soft_r2i_correct_mass_mean", ""),
        "soft_r2i_entropy_mean": last_stats.get("soft_r2i_entropy_mean", ""),
        "soft_i2r_top1_acc": last_stats.get("soft_i2r_top1_acc", ""),
        "soft_i2r_topk_acc": last_stats.get("soft_i2r_topk_acc", ""),
        "soft_i2r_correct_mass_mean": last_stats.get("soft_i2r_correct_mass_mean", ""),
        "soft_i2r_entropy_mean": last_stats.get("soft_i2r_entropy_mean", ""),
    })

out = Path("/kaggle/working/step8b_t4x2_soft_relation_summary.csv")
with out.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(out.read_text())

## 8. Archive artifacts

In [ ]:
from pathlib import Path
import json, shutil, subprocess

repo_dir = Path(CFG["WORK_DIR"]) / CFG["REPO_NAME"]
wsl_dir = repo_dir / "WSL_ReID"
info = json.loads(Path("/kaggle/working/step8b_t4x2_runtime_info.json").read_text())

artifact_dir = Path("/kaggle/working/artifacts_step8b_t4x2_softdiag")
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
(artifact_dir / "logs").mkdir(parents=True, exist_ok=True)
(artifact_dir / "relation_stats").mkdir(parents=True, exist_ok=True)

for p in [
    Path("/kaggle/working/step8b_t4x2_runtime_info.json"),
    Path("/kaggle/working/step8b_t4x2_soft_relation_summary.csv"),
]:
    if p.exists():
        shutil.copy2(p, artifact_dir / p.name)

for job in info["jobs"]:
    run_dir = wsl_dir.parent / "saved_regdb_resnet" / f"{job['save_path']}_{CFG['TRIAL']}"
    log_file = run_dir / "log" / "log.txt"
    if log_file.exists():
        shutil.copy2(log_file, artifact_dir / "logs" / f"{job['save_path']}_log.txt")
    stats_csv = run_dir / "relation_stats" / "relation_stats_summary.csv"
    if stats_csv.exists():
        shutil.copy2(stats_csv, artifact_dir / "relation_stats" / f"{job['save_path']}_relation_stats_summary.csv")

tar_path = Path("/kaggle/working/artifacts_step8b_t4x2_softdiag.tar.gz")
if tar_path.exists():
    tar_path.unlink()
subprocess.run(["tar", "-czf", str(tar_path), "-C", "/kaggle/working", artifact_dir.name], check=True)
print("Artifact:", tar_path)
print("Size MB:", tar_path.stat().st_size / (1024 * 1024))

## 9. Optional: push small backup to GitHub with old git push method

In [ ]:
if not CFG.get("PUSH_SMALL_BACKUP_TO_GIT", False):
    print("Git backup disabled.")
else:
    from kaggle_secrets import UserSecretsClient
    import os, subprocess, textwrap, shutil, json
    from pathlib import Path

    owner = CFG["GITHUB_OWNER"]
    repo_name = CFG["REPO_NAME"]
    branch = CFG["BRANCH"]
    repo_dir = Path(CFG["WORK_DIR"]) / repo_name
    token = UserSecretsClient().get_secret(CFG["GITHUB_TOKEN_SECRET"]).strip()

    info = json.loads(Path("/kaggle/working/step8b_t4x2_runtime_info.json").read_text())
    run_id = info["run_suffix"]
    backup_dir = repo_dir / "experiments" / "kaggle_runs" / run_id
    if backup_dir.exists():
        shutil.rmtree(backup_dir)
    backup_dir.mkdir(parents=True, exist_ok=True)

    artifact_dir = Path("/kaggle/working/artifacts_step8b_t4x2_softdiag")
    for item in artifact_dir.iterdir():
        dst = backup_dir / item.name
        if item.is_dir():
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

    (backup_dir / "README.md").write_text(f"""# Kaggle run backup: {run_id}\n\nStep 8B T4x2 soft relation diagnostics.\n\nThis directory contains small logs, relation stats, and summary CSV only.\n""")

    askpass = Path(CFG["WORK_DIR"]) / "git_askpass_push.sh"
    askpass.write_text(textwrap.dedent(f"""\
#!/bin/sh
case "$1" in
  *Username*) echo "{owner}" ;;
  *Password*) echo "{token}" ;;
  *) echo "" ;;
esac
"""))
    askpass.chmod(0o700)
    env = os.environ.copy()
    env["GIT_ASKPASS"] = str(askpass)
    env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        subprocess.run(["git", "config", "user.name", CFG["GIT_USER_NAME"]], cwd=repo_dir, check=True, env=env)
        subprocess.run(["git", "config", "user.email", CFG["GIT_USER_EMAIL"]], cwd=repo_dir, check=True, env=env)
        subprocess.run(["git", "fetch", "origin", branch], cwd=repo_dir, check=True, env=env)
        subprocess.run(["git", "checkout", branch], cwd=repo_dir, check=True, env=env)
        subprocess.run(["git", "pull", "--rebase", "origin", branch], cwd=repo_dir, check=True, env=env)
        rel = backup_dir.relative_to(repo_dir)
        subprocess.run(["git", "add", "-f", str(rel)], cwd=repo_dir, check=True, env=env)
        diff = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=repo_dir, env=env)
        if diff.returncode == 0:
            print("No staged changes.")
        else:
            subprocess.run(["git", "commit", "-m", f"exp: add Step 8B soft relation diagnostics {run_id}"], cwd=repo_dir, check=True, env=env)
            subprocess.run(["git", "push", "origin", branch], cwd=repo_dir, check=True, env=env)
        print(f"Pushed to https://github.com/{owner}/{repo_name}/tree/{branch}/experiments/kaggle_runs/{run_id}")
    finally:
        try:
            askpass.unlink()
        except FileNotFoundError:
            pass

In [ ]:
%%bash
cd /kaggle/working/UIT.CS2309/WSL_ReID

CUDA_VISIBLE_DEVICES=0 python main.py \
  --dataset regdb \
  --data-path /kaggle/working/VIREID_Dataset \
  --debug wsl \
  --save-path upr_v02_softdiag_filter055_p2s3_debug \
  --arch resnet \
  --trial 1 \
  --seed 1 \
  --model-path /kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth \
  --stage1-epoch 0 \
  --stage2-epoch 3 \
  --lr 0.00045 \
  --batch-pidnum 5 \
  --pid-numsample 4 \
  --test-batch 64 \
  --num-workers 2 \
  --device 0 \
  --milestones 8 12 \
  --upr-cre \
  --upr-beta 0.1 \
  --upr-gamma 0.0 \
  --upr-margin-weight 1.0 \
  --upr-warmup-epoch 2 \
  --upr-filter \
  --upr-filter-start-epoch 2 \
  --upr-filter-end-epoch 10 \
  --upr-filter-start-ratio 0.55 \
  --upr-filter-end-ratio 1.0 \
  --upr-filter-min-pairs 40 \
  --upr-soft-rel \
  --upr-soft-topk 3 \
  --upr-soft-temp 0.5 \
  --upr-soft-start-epoch 1 \
  --save-relation-stats \
  --relation-stats-dir ../saved_regdb_resnet/upr_v02_softdiag_filter055_p2s3_debug_1/relation_stats \
  > /kaggle/working/upr_v02_softdiag_filter055_p2s3_debug.log 2>&1

In [ ]:
%%bash
STATS_DIR="../saved_regdb_resnet/upr_v02_softdiag_filter055_p2s3_debug_1/relation_stats"

python scripts/collect_relation_stats.py \
  --stats-dir "$STATS_DIR" \
  --csv-output "$STATS_DIR/relation_stats_summary.csv"

cat "$STATS_DIR/relation_stats_summary.csv"
tail -80 /kaggle/working/upr_v02_softdiag_filter055_p2s3_debug.log

In [ ]:
%%bash
cd /kaggle/working/UIT.CS2309/WSL_ReID

STATS_DIR="../saved_regdb_resnet/upr_v02_softdiag_filter055_p2s3_debug_1/relation_stats"

echo "PWD=$(pwd)"
echo "STATS_DIR=$STATS_DIR"
ls -lah "$STATS_DIR"

python scripts/collect_relation_stats.py \
  --stats-dir "$STATS_DIR" \
  --csv-output "$STATS_DIR/relation_stats_summary.csv"

echo "===== relation_stats_summary.csv ====="
cat "$STATS_DIR/relation_stats_summary.csv"

## Step 8B-clean — chạy lại một run sạch baseline vs UPR softdiag 

In [6]:
%%bash
cd /kaggle/working/UIT.CS2309/WSL_ReID
mkdir -p /kaggle/working/run_logs

PHASE1_CKPT="/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth"

if [ ! -f "$PHASE1_CKPT" ]; then
  echo "ERROR: phase1 checkpoint not found: $PHASE1_CKPT"
  exit 1
fi

COMMIT=$(git rev-parse --short HEAD)
RUN_SUFFIX="step8bclean_$(date -u +%Y%m%d_%H%M%S)_${COMMIT}"

echo "COMMIT=$COMMIT"
echo "RUN_SUFFIX=$RUN_SUFFIX"
echo "PHASE1_CKPT=$PHASE1_CKPT"

CUDA_VISIBLE_DEVICES=0 nohup python main.py \
  --dataset regdb \
  --data-path /kaggle/working/VIREID_Dataset \
  --debug wsl \
  --save-path "baseline_softdiag_p2s5_${RUN_SUFFIX}" \
  --arch resnet \
  --trial 1 \
  --seed 1 \
  --model-path "$PHASE1_CKPT" \
  --stage1-epoch 0 \
  --stage2-epoch 5 \
  --lr 0.00045 \
  --batch-pidnum 5 \
  --pid-numsample 4 \
  --test-batch 64 \
  --num-workers 2 \
  --device 0 \
  --milestones 8 12 \
  --upr-soft-rel \
  --upr-soft-topk 3 \
  --upr-soft-temp 0.5 \
  --upr-soft-start-epoch 2 \
  --save-relation-stats \
  --relation-stats-dir "../saved_regdb_resnet/baseline_softdiag_p2s5_${RUN_SUFFIX}_1/relation_stats" \
  > "/kaggle/working/run_logs/baseline_softdiag_p2s5_${RUN_SUFFIX}.log" 2>&1 &

CUDA_VISIBLE_DEVICES=1 nohup python main.py \
  --dataset regdb \
  --data-path /kaggle/working/VIREID_Dataset \
  --debug wsl \
  --save-path "upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}" \
  --arch resnet \
  --trial 1 \
  --seed 1 \
  --model-path "$PHASE1_CKPT" \
  --stage1-epoch 0 \
  --stage2-epoch 5 \
  --lr 0.00045 \
  --batch-pidnum 5 \
  --pid-numsample 4 \
  --test-batch 64 \
  --num-workers 2 \
  --device 0 \
  --milestones 8 12 \
  --upr-cre \
  --upr-beta 0.1 \
  --upr-gamma 0.0 \
  --upr-margin-weight 1.0 \
  --upr-warmup-epoch 2 \
  --upr-filter \
  --upr-filter-start-epoch 2 \
  --upr-filter-end-epoch 10 \
  --upr-filter-start-ratio 0.55 \
  --upr-filter-end-ratio 1.0 \
  --upr-filter-min-pairs 40 \
  --upr-soft-rel \
  --upr-soft-topk 3 \
  --upr-soft-temp 0.5 \
  --upr-soft-start-epoch 2 \
  --save-relation-stats \
  --relation-stats-dir "../saved_regdb_resnet/upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}_1/relation_stats" \
  > "/kaggle/working/run_logs/upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}.log" 2>&1 &

echo "$RUN_SUFFIX" > /kaggle/working/step8b_clean_run_suffix.txt

echo "Started jobs:"
jobs

COMMIT=6b875c6
RUN_SUFFIX=step8bclean_20260625_031927_6b875c6
PHASE1_CKPT=/kaggle/input/datasets/truongtranmmcii/uit-cs2309-checkpoint/phase1_model_5.pth
Started jobs:
[1]-  Running                 CUDA_VISIBLE_DEVICES=0 nohup python main.py --dataset regdb --data-path /kaggle/working/VIREID_Dataset --debug wsl --save-path "baseline_softdiag_p2s5_${RUN_SUFFIX}" --arch resnet --trial 1 --seed 1 --model-path "$PHASE1_CKPT" --stage1-epoch 0 --stage2-epoch 5 --lr 0.00045 --batch-pidnum 5 --pid-numsample 4 --test-batch 64 --num-workers 2 --device 0 --milestones 8 12 --upr-soft-rel --upr-soft-topk 3 --upr-soft-temp 0.5 --upr-soft-start-epoch 2 --save-relation-stats --relation-stats-dir "../saved_regdb_resnet/baseline_softdiag_p2s5_${RUN_SUFFIX}_1/relation_stats" > "/kaggle/working/run_logs/baseline_softdiag_p2s5_${RUN_SUFFIX}.log" 2>&1 &
[2]+  Running                 CUDA_VISIBLE_DEVICES=1 nohup python main.py --dataset regdb --data-path /kaggle/working/VIREID_Dataset --debug wsl --save-path

In [19]:
%%bash
ps -ef | grep "python main.py" | grep -v grep || true
nvidia-smi
ls -lt /kaggle/working/run_logs | head -10
tail -40 /kaggle/working/run_logs/*.log

Thu Jun 25 03:36:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             65W /   70W |    8567MiB /  15360MiB |     93%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

tail: option used in invalid context -- 4


CalledProcessError: Command 'b'ps -ef | grep "python main.py" | grep -v grep || true\nnvidia-smi\nls -lt /kaggle/working/run_logs | head -10\ntail -40 /kaggle/working/run_logs/*.log\n'' returned non-zero exit status 1.

In [20]:
%%bash
RUN_SUFFIX=$(cat /kaggle/working/step8b_clean_run_suffix.txt)

echo "RUN_SUFFIX=$RUN_SUFFIX"

ps -ef | grep "python main.py" | grep -v grep || true
nvidia-smi

echo "===== baseline log ====="
tail -40 "/kaggle/working/run_logs/baseline_softdiag_p2s5_${RUN_SUFFIX}.log"

echo "===== UPR log ====="
tail -40 "/kaggle/working/run_logs/upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}.log"

RUN_SUFFIX=step8bclean_20260625_031927_6b875c6
Thu Jun 25 03:36:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             67W /   70W |    8567MiB /  15360MiB |    100%      Default |
|                                         |                        |                  N/A |
+

In [13]:
%%bash
nvidia-smi
RUN_SUFFIX=$(cat /kaggle/working/step8b_clean_run_suffix.txt)
tail -40 /kaggle/working/run_logs/baseline_softdiag_p2s5_${RUN_SUFFIX}.log
tail -40 /kaggle/working/run_logs/upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}.log

Thu Jun 25 03:23:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             65W /   70W |    8567MiB /  15360MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
%%bash
nvidia-smi --query-compute-apps=pid --format=csv,noheader | tr -d ' ' > /kaggle/working/step8b_current_pids.txt

echo "Tracking PIDs:"
cat /kaggle/working/step8b_current_pids.txt

while true; do
  RUNNING=0

  while read -r PID; do
    [ -n "$PID" ] || continue

    if kill -0 "$PID" 2>/dev/null; then
      echo "$(date -u '+%H:%M:%S') still running: PID=$PID"
      RUNNING=1
    else
      echo "$(date -u '+%H:%M:%S') finished: PID=$PID"
    fi
  done < /kaggle/working/step8b_current_pids.txt

  if [ "$RUNNING" -eq 0 ]; then
    echo "All jobs finished."
    break
  fi

  sleep 60
done

Tracking PIDs:
145
146
03:47:42 still running: PID=145
03:47:42 still running: PID=146
03:48:42 still running: PID=145
03:48:42 still running: PID=146
03:49:42 still running: PID=145
03:49:42 still running: PID=146
03:50:42 still running: PID=145
03:50:42 still running: PID=146
03:51:42 still running: PID=145
03:51:42 still running: PID=146
03:52:42 still running: PID=145
03:52:42 still running: PID=146
03:53:42 still running: PID=145
03:53:42 still running: PID=146
03:54:42 still running: PID=145
03:54:42 still running: PID=146
03:55:42 still running: PID=145
03:55:42 still running: PID=146
03:56:42 still running: PID=145
03:56:42 still running: PID=146
03:57:42 still running: PID=145
03:57:43 still running: PID=146
03:58:43 still running: PID=145
03:58:43 still running: PID=146
03:59:43 still running: PID=145
03:59:43 still running: PID=146
04:00:43 still running: PID=145
04:00:43 still running: PID=146
04:01:43 still running: PID=145
04:01:43 still running: PID=146
04:02:43 still ru

In [24]:
%%bash
set -e

cd /kaggle/working/UIT.CS2309/WSL_ReID

RUN_SUFFIX=$(cat /kaggle/working/step8b_clean_run_suffix.txt)

for RUN in \
  "baseline_softdiag_p2s5_${RUN_SUFFIX}_1" \
  "upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}_1"
do
  STATS_DIR="../saved_regdb_resnet/${RUN}/relation_stats"

  echo "=============================="
  echo "Collecting $RUN"
  echo "STATS_DIR=$STATS_DIR"

  python scripts/collect_relation_stats.py \
    --stats-dir "$STATS_DIR" \
    --csv-output "$STATS_DIR/relation_stats_summary.csv"

  echo "---- metrics ----"
  grep -E "R1:|Best_R1|Epoch:" "../saved_regdb_resnet/${RUN}/log/log.txt" | tail -60

  echo "---- relation summary ----"
  cat "$STATS_DIR/relation_stats_summary.csv"
done

STATS_DIR=../saved_regdb_resnet/baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1/relation_stats
../saved_regdb_resnet/baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1/relation_stats/relation_stats_summary.csv
---- metrics ----
Epoch: 0;Time: 2026-06-25 03:31:35;Setting: ../saved_regdb_resnet/baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1
R1:0.2694;   R10:0.4461;  R20:0.5311;  mAP:0.2799;  mINP:0.2103
                   Best_R1: 0.2694;    Best mAP: 0.2799
Epoch: 1;Time: 2026-06-25 03:42:35;Setting: ../saved_regdb_resnet/baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1
R1:0.3374;   R10:0.4903;  R20:0.5932;  mAP:0.3359;  mINP:0.2534
                   Best_R1: 0.3374;    Best mAP: 0.3359
Epoch: 2;Time: 2026-06-25 03:53:37;Setting: ../saved_regdb_resnet/baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1
R1:0.3728;   R10:0.5607;  R20:0.6335;  mAP:0.3812;  mINP:0.2935
                   Best_R1: 0.3728;    Best mAP: 0.3812
Epoch: 3

In [25]:
%%bash
cd /kaggle/working/UIT.CS2309/WSL_ReID

python - <<'PY'
import csv
import re
from pathlib import Path

run_suffix = Path("/kaggle/working/step8b_clean_run_suffix.txt").read_text().strip()
base = Path("../saved_regdb_resnet")

runs = [
    ("baseline_softdiag", f"baseline_softdiag_p2s5_{run_suffix}_1"),
    ("upr_v02_filter055_softdiag", f"upr_v02_softdiag_filter055_p2s5_{run_suffix}_1"),
]

rows = []

def parse_best_metrics(log_text):
    matches = re.findall(
        r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)",
        log_text,
    )
    if not matches:
        return "", "", ""
    best = max(matches, key=lambda x: float(x[3]))
    return best[0], best[3], best[4]

for tag, run in runs:
    log_path = base / run / "log" / "log.txt"
    stats_path = base / run / "relation_stats" / "relation_stats_summary.csv"

    log_text = log_path.read_text() if log_path.exists() else ""
    best_r1, best_map, best_minp = parse_best_metrics(log_text)

    last_stats = {}
    if stats_path.exists():
        with stats_path.open() as f:
            reader = list(csv.DictReader(f))
            if reader:
                last_stats = reader[-1]

    rows.append({
        "tag": tag,
        "run": run,
        "best_r1_by_map": best_r1,
        "best_map": best_map,
        "best_minp": best_minp,
        "last_common_accuracy": last_stats.get("common_accuracy", ""),
        "last_mutual_match_ratio": last_stats.get("mutual_match_ratio", ""),
        "soft_r2i_top1_acc": last_stats.get("soft_r2i_top1_acc", ""),
        "soft_r2i_topk_acc": last_stats.get("soft_r2i_topk_acc", ""),
        "soft_r2i_correct_mass_mean": last_stats.get("soft_r2i_correct_mass_mean", ""),
        "soft_r2i_entropy_mean": last_stats.get("soft_r2i_entropy_mean", ""),
        "soft_i2r_top1_acc": last_stats.get("soft_i2r_top1_acc", ""),
        "soft_i2r_topk_acc": last_stats.get("soft_i2r_topk_acc", ""),
        "soft_i2r_correct_mass_mean": last_stats.get("soft_i2r_correct_mass_mean", ""),
        "soft_i2r_entropy_mean": last_stats.get("soft_i2r_entropy_mean", ""),
    })

out = Path("/kaggle/working/step8b_clean_soft_relation_summary.csv")
with out.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(out.read_text())
PY

tag,run,best_r1_by_map,best_map,best_minp,last_common_accuracy,last_mutual_match_ratio,soft_r2i_top1_acc,soft_r2i_topk_acc,soft_r2i_correct_mass_mean,soft_r2i_entropy_mean,soft_i2r_top1_acc,soft_i2r_topk_acc,soft_i2r_correct_mass_mean,soft_i2r_entropy_mean
baseline_softdiag,baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1,0.4539,0.4459,0.3343,0.6153846153846154,0.8719512195121951,0.47572815533980584,0.6262135922330098,0.3814004940110767,0.7222031821739662,0.5,0.6116504854368932,0.3916901148024065,0.713809852646758
upr_v02_filter055_softdiag,upr_v02_softdiag_filter055_p2s5_step8bclean_20260625_031927_6b875c6_1,0.4214,0.4064,0.3004,0.6043956043956044,0.9381443298969072,0.4029126213592233,0.5485436893203883,0.3027285191746376,0.8134318561148212,0.4368932038834951,0.5922330097087378,0.3412463687492416,0.7844595540274345



In [26]:
%%bash
set -e

cd /kaggle/working/UIT.CS2309/WSL_ReID

RUN_SUFFIX=$(cat /kaggle/working/step8b_clean_run_suffix.txt)

ART_DIR="/kaggle/working/artifacts_step8b_clean_${RUN_SUFFIX}"
mkdir -p "$ART_DIR/logs" "$ART_DIR/relation_stats"

cp /kaggle/working/step8b_clean_soft_relation_summary.csv "$ART_DIR/" || true

for RUN in \
  "baseline_softdiag_p2s5_${RUN_SUFFIX}_1" \
  "upr_v02_softdiag_filter055_p2s5_${RUN_SUFFIX}_1"
do
  cp "../saved_regdb_resnet/${RUN}/log/log.txt" \
     "$ART_DIR/logs/${RUN}_log.txt" || true

  cp "../saved_regdb_resnet/${RUN}/relation_stats/relation_stats_summary.csv" \
     "$ART_DIR/relation_stats/${RUN}_relation_stats_summary.csv" || true
done

tar -czf "/kaggle/working/artifacts_step8b_clean_${RUN_SUFFIX}.tar.gz" \
  -C /kaggle/working "$(basename "$ART_DIR")"

ls -lh "/kaggle/working/artifacts_step8b_clean_${RUN_SUFFIX}.tar.gz"

-rw-r--r-- 1 root root 4.0K Jun 25 04:17 /kaggle/working/artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6.tar.gz


In [32]:
%%bash
pwd
cp -r -v /artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6 /UIT.CS2309/experiments/kaggle_runs

/kaggle/working


cp: cannot stat '/artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6': No such file or directory
bash: line 3: cd: /UIT.CS2309: No such file or directory
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


CalledProcessError: Command 'b'pwd\ncp -r -v /artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6 /UIT.CS2309/experiments/kaggle_runs\ncd /UIT.CS2309\ngit add .\ngit commit -m "step 8 clean"\ngit push\n'' returned non-zero exit status 128.

In [35]:
%%bash
cd UIT.CS2309
pwd
git config --global user.email "truongtlt147@gmail.com"
git config --global user.name "TranTruongMMCII"
git add .
git commit -m "step 8 clean"
git push

/kaggle/working/UIT.CS2309
[feature/upr-cre 7ac0afb] step 8 clean
 3 files changed, 15 insertions(+)
 create mode 100644 experiments/kaggle_runs/artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6/relation_stats/baseline_softdiag_p2s5_step8bclean_20260625_031927_6b875c6_1_relation_stats_summary.csv
 create mode 100644 experiments/kaggle_runs/artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6/relation_stats/upr_v02_softdiag_filter055_p2s5_step8bclean_20260625_031927_6b875c6_1_relation_stats_summary.csv
 create mode 100644 experiments/kaggle_runs/artifacts_step8b_clean_step8bclean_20260625_031927_6b875c6/step8b_clean_soft_relation_summary.csv


fatal: could not read Username for 'https://github.com': No such device or address


CalledProcessError: Command 'b'cd UIT.CS2309\npwd\ngit config --global user.email "truongtlt147@gmail.com"\ngit config --global user.name "TranTruongMMCII"\ngit add .\ngit commit -m "step 8 clean"\ngit push\n'' returned non-zero exit status 128.